In [2]:
import pandas as pd

In [3]:
df_train_raw = pd.read_csv(
    "data/landing/sales_train.csv", parse_dates=["date"], date_format="%d.%m.%Y"
).sort_values(by="date", ignore_index=True)
df_train_raw["date_block_num"] = df_train_raw["date_block_num"].astype("int16")
df_train_raw["shop_id"] = df_train_raw["shop_id"].astype("int16")
df_train_raw["item_id"] = df_train_raw["item_id"].astype("int16")
df_train_raw["item_price"] = df_train_raw["item_price"].astype("float32")
df_train_raw["item_cnt_day"] = df_train_raw["item_cnt_day"].astype("int16")
df_train_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2935849 entries, 0 to 2935848
Data columns (total 6 columns):
 #   Column          Dtype         
---  ------          -----         
 0   date            datetime64[us]
 1   date_block_num  int16         
 2   shop_id         int16         
 3   item_id         int16         
 4   item_price      float32       
 5   item_cnt_day    int16         
dtypes: datetime64[us](1), float32(1), int16(4)
memory usage: 56.0 MB


## Чистка до аггрегации

In [4]:
df_train = df_train_raw.copy()

In [5]:
df_train = df_train.drop_duplicates(subset=['date', 'shop_id', 'item_id'], ignore_index=True)

In [6]:
df_train['is_refund'] = (df_train_raw['item_cnt_day'] < 0).astype(bool)
df_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,is_refund
0,2013-01-01,0,18,5823,2500.0,1,False
1,2013-01-01,0,27,5573,849.0,1,False
2,2013-01-01,0,7,1006,399.0,1,False
3,2013-01-01,0,19,17707,899.0,1,False
4,2013-01-01,0,14,19548,149.0,1,False


In [7]:
shop_map = {0: 57, 1: 58, 11: 10}
df_train['shop_id'] = df_train['shop_id'].replace(shop_map)
print(sorted(df_train['shop_id'].unique().tolist()));

[2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]


## Разбивка

In [8]:
tmp_df = df_train[df_train.item_cnt_day > 0]
tables = [g for _, g in tmp_df.groupby('date_block_num', sort=True)]
init_date_block_num = tmp_df['date_block_num'].iloc[0]
table = tables[0]
table.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,is_refund
0,2013-01-01,0,18,5823,2500.0,1,False
1,2013-01-01,0,27,5573,849.0,1,False
2,2013-01-01,0,7,1006,399.0,1,False
3,2013-01-01,0,19,17707,899.0,1,False
4,2013-01-01,0,14,19548,149.0,1,False


In [9]:
parts = []
keys = ["date_block_num", "shop_id", "item_id"]
for i, table in enumerate(tables):
    long = (
        table.pivot_table(index="shop_id", columns="item_id",
                          values="item_cnt_day", aggfunc="sum")
        .fillna(0)
        .reset_index()
        .melt(id_vars="shop_id", value_name="target")
        .assign(date_block_num=i + init_date_block_num)
    )
    parts.append(long)

df = pd.concat(parts, ignore_index=True).sort_values(keys)
df.head()

,shop_id,item_id,target,date_block_num
0,2,19,0.0,0
45,2,27,1.0,0
90,2,28,0.0,0
135,2,29,0.0,0
180,2,32,0.0,0


## Тестовый вариант (Claude)

In [10]:
keys = ["date_block_num", "shop_id", "item_id"]
pos = df_train[df_train.item_cnt_day > 0]

parts = []
for m, g in pos.groupby("date_block_num"):
    shops, items = g.shop_id.unique(), g.item_id.unique()
    parts.append(pd.MultiIndex.from_product([[m], shops, items], names=keys).to_frame(index=False)) # type: ignore
grid = pd.concat(parts, ignore_index=True) # type: ignore

agg = df_train.groupby(keys, as_index=False).item_cnt_day.sum()
new_df = grid.merge(agg, on=keys, how="left")
new_df["target"] = new_df.pop("item_cnt_day").fillna(0).clip(0, 20)
new_df = new_df.sort_values(keys)
new_df.head()

,date_block_num,shop_id,item_id,target
136379,0,2,19,0.0
130904,0,2,27,1.0
132480,0,2,28,0.0
134497,0,2,29,0.0
129922,0,2,32,0.0


In [11]:
new_df.target.describe()

count    1.088455e+07
mean     2.990403e-01
std      1.223914e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.000000e+01
Name: target, dtype: float64